# DuckDB & Parquet - Hive Partitioning

The partition based filters help only inspecting the required files (aka "predicate pushdown").

In [1]:
%config SqlMagic.displaylimit = 100
%config SqlMagic.autopandas = False

In [2]:
%%capture
%load_ext sql

In [3]:
# https://duckdb.org/docs/lts/guides/python/jupyter
%sql duckdb:///:memory: --alias duckdb

Connecting to 'duckdb'

💡 The [`EXPLAIN ANALYZE`](https://duckdb.org/docs/current/guides/meta/explain_analyze) profiling query is executed through the Python client for a nicer output in the notebook.

💡 Comment either the filter `AND year = 2023` or `AND month = 3` to see how the predicate pushdown works.

In [4]:
import duckdb

with duckdb.connect() as _conn:
    # https://duckdb.org/docs/current/guides/meta/explain_analyze
    # https://duckdb.org/docs/current/clients/python/known_issues
    print(_conn.sql("""
    PRAGMA enable_profiling;

    SELECT
        species,
        avg(body_mass_g) AS body_mass_g,
    FROM read_parquet('../data/penguins/*/*/*.parquet', hive_partitioning = true)
    WHERE
        true
        AND year = 2023
        AND month = 3
    GROUP BY ALL;
    """).explain('analyze'))

╭─ Summary ───────────╮
│ Total Time: 0.0086s │
│ Data Read:   9.7 kB │
╰─────────────────────╯
╭─ Hash Group By ───────────────────────────────────╮
│ Groups: #0                                        │
│ Aggregates: avg(#1)                               │
│ 3 rows                                        0µs │
╰─────────────────────────┒┎────────────────────────╯
╭─ Projection ────────────┚┖────────────────────────╮
│ Projections: species, body_mass_g                 │
│ 133 rows                                      0µs │
╰─────────────────────────┒┎────────────────────────╯
╭─ Table Scan ────────────┚┖────────────────────────╮
│ Function: READ_PARQUET                            │
│ Projections: species, body_mass_g                 │
│ File Filters: (year = 2023)(month = 3)            │
│ Scanning Files: 1/72                              │
│ Total Files Read: 1                               │
│ Filename(s):                                      │
│ ../data/penguins/year=2023/month=3/dat

Only the file `../data/penguins/year=2023/month=3/data_0.parquet` is accessed, with respect to the SQL filter `year = 2023 AND month = 3` (block `TABLE_SCAN` for the function `READ_PARQUET`).